# nnU-Net 3D full resolution + soft-clDice (v2)

## Experimento 2b — ImageCAS

**Objetivo:** repetir el experimento de soft-clDice incorporando dos correcciones metodológicas identificadas al analizar el entrenamiento de v1:

1. **clDice solo en resolución completa.** En v1, `DeepSupervisionWrapper` envolvía toda la pérdida combinada, de modo que el término topológico también se calculaba en las escalas submuestreadas de deep supervision, donde la coronaria (1-3 vóxeles de grosor) casi desaparece tras el remuestreo `nearest`. En v2 el clDice se calcula una única vez, sobre `outputs[0]` (salida de resolución completa).
2. **Warm-up lineal de `lambda_cldice`.** En v1, `lambda_cldice = 0.2` era fijo desde la época 0, cuando las predicciones son prácticamente aleatorias y el esqueleto es inestable (posible causa de la caída inicial observada en el log de v1: 0.4659 frente a 0.5366 del baseline en la época 0). En v2, `lambda_cldice` crece linealmente de 0 a 0.2 durante las primeras 10 épocas.

Configuración (idéntica a v1 para garantizar la comparabilidad):
- Dataset: Dataset501_ImageCAS
- Configuración: 3d_fullres
- Fold: 0
- Épocas: 100
- Clase del trainer: `nnUNetTrainerSoftClDiceV2_100epochs` (nombre distinto del de v1 para no sobrescribir sus resultados y permitir la comparación de los tres experimentos: baseline, clDice v1 y clDice v2)


In [1]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA disponible:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


PyTorch: 2.11.0+cu128
CUDA disponible: True
GPU: NVIDIA A100-SXM4-40GB


In [2]:
!python -m pip install -q nnunetv2


In [ ]:
import os
os.kill(os.getpid(), 9)


In [2]:
from google.colab import drive
drive.mount("/content/drive", force_remount=True)


Mounted at /content/drive


In [3]:
import os
from pathlib import Path

BASE = Path("/content/drive/MyDrive/TFM_nnUNet")

NNUNET_RAW = BASE / "nnUNet_raw"
NNUNET_PREPROCESSED = BASE / "nnUNet_preprocessed"
NNUNET_RESULTS = BASE / "nnUNet_results"

os.environ["nnUNet_raw"] = str(NNUNET_RAW)
os.environ["nnUNet_preprocessed"] = str(NNUNET_PREPROCESSED)
os.environ["nnUNet_results"] = str(NNUNET_RESULTS)

print("RAW:", NNUNET_RAW)
print("PREPROCESSED:", NNUNET_PREPROCESSED)
print("RESULTS:", NNUNET_RESULTS)


RAW: /content/drive/MyDrive/TFM_nnUNet/nnUNet_raw
PREPROCESSED: /content/drive/MyDrive/TFM_nnUNet/nnUNet_preprocessed
RESULTS: /content/drive/MyDrive/TFM_nnUNet/nnUNet_results


## 1. Operaciones morfológicas diferenciables 3D

Idénticas a las de v1, ya verificadas mediante el test de gradientes.


In [4]:
import torch
import torch.nn.functional as F


def soft_erode_3d(x):
    """
    Aproximacion diferenciable de una erosion 3D.
    """
    p1 = -F.max_pool3d(-x, kernel_size=(3, 1, 1), stride=1, padding=(1, 0, 0))
    p2 = -F.max_pool3d(-x, kernel_size=(1, 3, 1), stride=1, padding=(0, 1, 0))
    p3 = -F.max_pool3d(-x, kernel_size=(1, 1, 3), stride=1, padding=(0, 0, 1))
    return torch.minimum(torch.minimum(p1, p2), p3)


def soft_dilate_3d(x):
    return F.max_pool3d(x, kernel_size=3, stride=1, padding=1)


def soft_open_3d(x):
    return soft_dilate_3d(soft_erode_3d(x))


def soft_skeletonize_3d(x, iterations=10):
    opened = soft_open_3d(x)
    skeleton = F.relu(x - opened)

    for _ in range(iterations):
        x = soft_erode_3d(x)
        opened = soft_open_3d(x)
        delta = F.relu(x - opened)
        skeleton = skeleton + F.relu(delta - skeleton * delta)

    return skeleton


## 2. SoftClDiceLoss

Sin cambios respecto a v1.


In [5]:
import torch.nn as nn


class SoftClDiceLoss(nn.Module):

    def __init__(self, iterations=10, smooth=1e-5):
        super().__init__()
        self.iterations = iterations
        self.smooth = smooth

    def forward(self, prediction, target):
        prediction = prediction.float()
        target = target.float()

        skel_pred = soft_skeletonize_3d(prediction, iterations=self.iterations)
        skel_true = soft_skeletonize_3d(target, iterations=self.iterations)

        tprec = ((skel_pred * target).sum() + self.smooth) / (skel_pred.sum() + self.smooth)
        tsens = ((skel_true * prediction).sum() + self.smooth) / (skel_true.sum() + self.smooth)

        cldice = (2.0 * tprec * tsens) / (tprec + tsens + self.smooth)
        return 1.0 - cldice


## 3. Cambio 1 — Dice + CE (deep supervision) y soft-clDice solo en resolución completa

El término `regional_loss_ds` mantiene Dice + CE ponderado en todas las escalas de deep supervision, como en v1. El término `cldice_loss` se calcula fuera de ese wrapper y una sola vez, sobre el primer elemento de las listas de salidas y targets (`outputs[0]` / `targets[0]`), que en nnU-Net corresponde siempre a la resolución completa.


In [6]:
class DiceCEPlusSoftClDiceFullResOnly(nn.Module):

    def __init__(self, regional_loss_ds, cldice_loss, lambda_cldice=0.2):
        super().__init__()
        self.regional_loss_ds = regional_loss_ds
        self.cldice_loss = cldice_loss
        self.lambda_cldice = lambda_cldice  # se sobreescribe cada epoca (warm-up)

    def forward(self, outputs, targets):
        loss_regional = self.regional_loss_ds(outputs, targets)

        if isinstance(outputs, (list, tuple)):
            logits_fullres = outputs[0]
            target_fullres = targets[0]
        else:
            logits_fullres = outputs
            target_fullres = targets

        probabilities = torch.softmax(logits_fullres.float(), dim=1)
        foreground_probability = probabilities[:, 1:2]

        if target_fullres.ndim == logits_fullres.ndim - 1:
            target_fullres = target_fullres.unsqueeze(1)

        foreground_target = (target_fullres == 1).float()

        topology_loss = self.cldice_loss(foreground_probability, foreground_target)

        return loss_regional + self.lambda_cldice * topology_loss


## 4. Test de gradientes

Comprobación de que `SoftClDiceLoss` admite retropropagación. La pérdida de base es la misma que en v1.


In [7]:
device = "cuda" if torch.cuda.is_available() else "cpu"

pred_logits = torch.randn(1, 1, 24, 32, 32, device=device, requires_grad=True)
pred_probability = torch.sigmoid(pred_logits)

target_test = torch.zeros_like(pred_probability)
target_test[:, :, 4:20, 15:17, 15:17] = 1  # estructura tubular sintetica

test_loss = SoftClDiceLoss(iterations=5)(pred_probability, target_test)
print("soft-clDice loss:", test_loss.item())

test_loss.backward()
print("Gradiente calculado:", pred_logits.grad is not None)
print("Gradientes finitos:", torch.isfinite(pred_logits.grad).all().item())


soft-clDice loss: 0.9947634935379028
Gradiente calculado: True
Gradientes finitos: True


## 5. Cambio 2 — Verificación del warm-up de lambda_cldice

Comprobación previa al entrenamiento de que la rampa lineal de 0 a 0.2 en 10 épocas se comporta según lo previsto.


In [8]:
target_lambda_cldice = 0.2
warmup_epochs = 10

def lambda_at_epoch(epoch, target=target_lambda_cldice, warmup=warmup_epochs):
    if epoch < warmup:
        return target * (epoch / warmup)
    return target

for e in [0, 1, 2, 5, 9, 10, 11, 50, 99]:
    print(f"epoca {e:>3}: lambda_cldice = {lambda_at_epoch(e):.4f}")


epoca   0: lambda_cldice = 0.0000
epoca   1: lambda_cldice = 0.0200
epoca   2: lambda_cldice = 0.0400
epoca   5: lambda_cldice = 0.1000
epoca   9: lambda_cldice = 0.1800
epoca  10: lambda_cldice = 0.2000
epoca  11: lambda_cldice = 0.2000
epoca  50: lambda_cldice = 0.2000
epoca  99: lambda_cldice = 0.2000


## 6. Carpeta de trainers personalizados en Drive


In [9]:
from pathlib import Path

CUSTOM_TRAINERS = BASE / "custom_trainers"
CUSTOM_TRAINERS.mkdir(parents=True, exist_ok=True)

os.environ["nnUNet_extTrainer"] = str(CUSTOM_TRAINERS)

print("Carpeta trainers:", CUSTOM_TRAINERS)
print("nnUNet_extTrainer:", os.environ["nnUNet_extTrainer"])


Carpeta trainers: /content/drive/MyDrive/TFM_nnUNet/custom_trainers
nnUNet_extTrainer: /content/drive/MyDrive/TFM_nnUNet/custom_trainers


## 7. Registro del trainer completo (`nnUNetTrainerSoftClDiceV2_100epochs`)

Al igual que en v1, nnU-Net requiere que la clase se encuentre en un archivo `.py`. El trainer integra las dos modificaciones: `DiceCEPlusSoftClDiceFullResOnly` en `_build_loss()` y el warm-up de `lambda_cldice` mediante el hook `on_epoch_start`.

Nota: algunas versiones antiguas de nnU-Net denominan el hook `on_train_epoch_start`. Si la versión instalada no expone `on_epoch_start`, el warm-up no se aplica y `lambda_cldice` permanece en 0. Por ello, el log de entrenamiento debe mostrar líneas `"lambda_cldice actual: ..."` con valores crecientes durante las primeras 10 épocas.


In [10]:
trainer_code = r'''
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

from nnunetv2.training.loss.compound_losses import DC_and_CE_loss
from nnunetv2.training.loss.deep_supervision import DeepSupervisionWrapper
from nnunetv2.training.loss.dice import MemoryEfficientSoftDiceLoss

from nnunetv2.training.nnUNetTrainer.variants.training_length.nnUNetTrainer_Xepochs import (
    nnUNetTrainer_100epochs
)


def soft_erode_3d(x):
    p1 = -F.max_pool3d(-x, kernel_size=(3, 1, 1), stride=1, padding=(1, 0, 0))
    p2 = -F.max_pool3d(-x, kernel_size=(1, 3, 1), stride=1, padding=(0, 1, 0))
    p3 = -F.max_pool3d(-x, kernel_size=(1, 1, 3), stride=1, padding=(0, 0, 1))
    return torch.minimum(torch.minimum(p1, p2), p3)


def soft_dilate_3d(x):
    return F.max_pool3d(x, kernel_size=3, stride=1, padding=1)


def soft_open_3d(x):
    return soft_dilate_3d(soft_erode_3d(x))


def soft_skeletonize_3d(x, iterations=10):
    opened = soft_open_3d(x)
    skeleton = F.relu(x - opened)

    for _ in range(iterations):
        x = soft_erode_3d(x)
        opened = soft_open_3d(x)
        delta = F.relu(x - opened)
        skeleton = skeleton + F.relu(delta - skeleton * delta)

    return skeleton


class SoftClDiceLoss(nn.Module):

    def __init__(self, iterations=10, smooth=1e-5):
        super().__init__()
        self.iterations = iterations
        self.smooth = smooth

    def forward(self, prediction, target):
        prediction = prediction.float()
        target = target.float()

        skel_pred = soft_skeletonize_3d(prediction, iterations=self.iterations)
        skel_true = soft_skeletonize_3d(target, iterations=self.iterations)

        tprec = ((skel_pred * target).sum() + self.smooth) / (skel_pred.sum() + self.smooth)
        tsens = ((skel_true * prediction).sum() + self.smooth) / (skel_true.sum() + self.smooth)

        cldice = (2.0 * tprec * tsens) / (tprec + tsens + self.smooth)
        return 1.0 - cldice


class DiceCEPlusSoftClDiceFullResOnly(nn.Module):

    def __init__(self, regional_loss_ds, cldice_loss, lambda_cldice=0.2):
        super().__init__()
        self.regional_loss_ds = regional_loss_ds
        self.cldice_loss = cldice_loss
        self.lambda_cldice = lambda_cldice

    def forward(self, outputs, targets):
        loss_regional = self.regional_loss_ds(outputs, targets)

        if isinstance(outputs, (list, tuple)):
            logits_fullres = outputs[0]
            target_fullres = targets[0]
        else:
            logits_fullres = outputs
            target_fullres = targets

        probabilities = torch.softmax(logits_fullres.float(), dim=1)
        foreground_probability = probabilities[:, 1:2]

        if target_fullres.ndim == logits_fullres.ndim - 1:
            target_fullres = target_fullres.unsqueeze(1)

        foreground_target = (target_fullres == 1).float()

        topology_loss = self.cldice_loss(foreground_probability, foreground_target)

        return loss_regional + self.lambda_cldice * topology_loss


class nnUNetTrainerSoftClDiceV2_100epochs(nnUNetTrainer_100epochs):

    def __init__(self, plans, configuration, fold, dataset_json, device=torch.device("cuda")):
        super().__init__(plans, configuration, fold, dataset_json, device)

        self.target_lambda_cldice = 0.2
        self.warmup_epochs = 10
        self.skeleton_iterations = 10

    def _build_loss(self):
        regional_loss = DC_and_CE_loss(
            soft_dice_kwargs={
                "batch_dice": self.configuration_manager.batch_dice,
                "smooth": 1e-5,
                "do_bg": False,
                "ddp": self.is_ddp
            },
            ce_kwargs={},
            weight_ce=1,
            weight_dice=1,
            ignore_label=self.label_manager.ignore_label,
            dice_class=MemoryEfficientSoftDiceLoss
        )

        if self.enable_deep_supervision:
            deep_supervision_scales = self._get_deep_supervision_scales()
            weights = np.array([1 / (2 ** i) for i in range(len(deep_supervision_scales))])
            weights[-1] = 0
            weights = weights / weights.sum()
            regional_loss_ds = DeepSupervisionWrapper(regional_loss, weights)
        else:
            regional_loss_ds = regional_loss

        cldice_loss = SoftClDiceLoss(iterations=self.skeleton_iterations, smooth=1e-5)

        loss = DiceCEPlusSoftClDiceFullResOnly(
            regional_loss_ds=regional_loss_ds,
            cldice_loss=cldice_loss,
            lambda_cldice=0.0
        )

        return loss

    def _get_current_lambda_cldice(self):
        if self.current_epoch < self.warmup_epochs:
            return self.target_lambda_cldice * (self.current_epoch / self.warmup_epochs)
        return self.target_lambda_cldice

    def on_epoch_start(self):
        super().on_epoch_start()
        self.loss.lambda_cldice = self._get_current_lambda_cldice()
        self.print_to_log_file(
            f"lambda_cldice actual: {self.loss.lambda_cldice:.4f}"
        )
'''

trainer_file = CUSTOM_TRAINERS / "nnUNetTrainerSoftClDiceV2_100epochs.py"
trainer_file.write_text(trainer_code)

print("Guardado en:")
print(trainer_file)


Guardado en:
/content/drive/MyDrive/TFM_nnUNet/custom_trainers/nnUNetTrainerSoftClDiceV2_100epochs.py


In [11]:
!echo $nnUNet_extTrainer


/content/drive/MyDrive/TFM_nnUNet/custom_trainers


## 8. Entrenamiento soft-clDice v2 (100 épocas)

- El nombre de clase (`...V2_100epochs`) difiere del de v1, de modo que los resultados se almacenan en una carpeta independiente.
- Ante una desconexión de Colab, el entrenamiento se reanuda con la opción `--c`:
  `!nnUNetv2_train 501 3d_fullres 0 -tr nnUNetTrainerSoftClDiceV2_100epochs --npz --c`


In [12]:
!nnUNetv2_train \
501 \
3d_fullres \
0 \
-tr nnUNetTrainerSoftClDiceV2_100epochs \
--npz



############################
INFO: You are using the old nnU-Net default plans. We have updated our recommendations. Please consider using those instead! Read more here: https://github.com/MIC-DKFZ/nnUNet/blob/master/documentation/resenc_presets.md
############################

Trainer 'nnUNetTrainerSoftClDiceV2_100epochs' not found in nnunetv2.training.nnUNetTrainer.
Searching in external trainer paths from environment variable 'nnUNet_extTrainer'...
Searching in: /content/drive/MyDrive/TFM_nnUNet/custom_trainers
Searching for class nnUNetTrainerSoftClDiceV2_100epochs in folder /content/drive/MyDrive/TFM_nnUNet/custom_trainers with current module None
  Inspecting module: nnUNetTrainerSoftClDice100epochs
  Inspecting module: nnUNetTrainerSoftClDiceV2_100epochs
Found class nnUNetTrainerSoftClDiceV2_100epochs in nnUNetTrainerSoftClDiceV2_100epochs
Using trainer 'nnUNetTrainerSoftClDiceV2_100epochs' from: /content/drive/MyDrive/TFM_nnUNet/custom_trainers
Using device: cuda:0

##########

In [13]:
!nvidia-smi

Thu Aug 13 19:27:41 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   38C    P0             56W /  400W |     524MiB /  40960MiB |     23%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----